# Lab 16: Regularization — Buying Bias to Sell Variance
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 47 min Core + 55 min AI + 20 min Extension + take-home Challenge

---

**Learning Objectives:**
- Show OLS failing when predictors outnumber useful signal
- Trace the coefficient path as the penalty grows
- Select a penalty by cross-validation, with the scaler fitted inside every fold, and check what LASSO kept
- Explain why the penalty introduces bias deliberately, and what it buys
- Write your own prompt for a cross-validation loop, and check the AI's loop against the lab's own penalty and CV error

**Dataset:** Simulated high-dimensional design with known sparse truth

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one. Lab 15's `KFold` loop is used again here.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 runs one guided step, then has two one-line blanks, in Step 9. The AI-Assisted Expansion is required: for the first time, you write the prompt yourself. The Extension is optional, and the Challenge is a take-home.

**Plan on about 2 hours** for the required work: setup, Parts 1-4, the AI-Assisted Expansion, and the README and submission.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 9:** two one-line blanks (`____`): the LASSO and Ridge fits on a design with two near-duplicate predictors.
> - The answers to the interpretation questions, in text cells: one question each after Parts 1, 2 and 4.
> - **The AI-Assisted Expansion:** your own prompt, from scratch, asking AI for a cross-validation loop over the penalty; a revised prompt for round 2 (and round 3 if you need it); a short checking log; and two answers. Your prompts, the log and the answers are what is graded, not the AI's code.
>
> **You run and read:** everything else. Setup, Parts 1-3 and Step 8 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The check cells in the AI section: run them; you need not follow their code. The AI's function is pasted, not typed.
>
> **The Extension** (optional) and **the Challenge** (take-home) have no starter code: you write them in new cells, from the task. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

The first line of the setup cell upgrades scikit-learn to 1.8 or later, because `LassoCV`
changed in 1.8 and Step 9's numbers come from the newer version. The cell prints your
scikit-learn version. If it shows a version below 1.8, use **Runtime → Restart session** and run this cell again.

Every random number in this lab comes from one generator, `rng`, and each draw moves it on. So
your numbers match the expected output only if you run the cells once, from the top, in order. If
you run a cell twice, use **Runtime → Restart session** and run from the top again.

In [ ]:
# GUIDED — run as-is
# Step 1: imports and the seed
!pip install "scikit-learn>=1.8" -q

import sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, RidgeCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import metrics
from pathlib import Path

rng = np.random.default_rng(seed=42)    # the one generator for every random draw
RANDOM_STATE = 42                       # the seed for the train-test split and the fold split

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print(f"scikit-learn {sklearn.__version__} (Step 9's numbers assume 1.8 or later)")
print("Setup complete.")

## Part 1: Where OLS Runs Out (GUIDED — 15 min)

Chapter 15 gave you a dial with a few settings. Here complexity means *which of
200 variables to include*, and there are more subsets than atoms in the
observable universe.

Step 2 builds a world where the truth is known: 150 rows and 200 predictors, each drawn from a
standard normal, and an outcome that depends on the first five only. `rng.normal(0, 1, (n, p))`
draws a table of n rows and p columns. `beta[:k] = [...]` sets the first five coefficients and
leaves the other 195 at zero. `X @ beta` is matrix multiplication: for each row, every predictor
times its coefficient, added up. Then a 70/30 train-test split, as in Lab 6, and OLS on the
training rows with `LinearRegression().fit(X, y)`, as in Lab 12.

In [ ]:
# GUIDED — run as-is
# Step 2: 200 predictors, 5 of which matter, and OLS on the training rows
n, p, k = 150, 200, 5
X = rng.normal(0, 1, (n, p))
beta = np.zeros(p)
beta[:k] = [3.0, -2.0, 1.5, 2.5, -1.8]     # only the first five are real
y = X @ beta + rng.normal(0, 1.0, n)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=RANDOM_STATE)
print(f"n = {n}, p = {p}, truly nonzero = {k}")
print(f"training rows = {len(X_tr)}, which is fewer than p = {p}")

ols = LinearRegression().fit(X_tr, y_tr)
print(f"\nOLS train MSE : {metrics.mean_squared_error(y_tr, ols.predict(X_tr)):.4f}")
print(f"OLS test  MSE : {metrics.mean_squared_error(y_te, ols.predict(X_te)):.4f}")

**Expected output:** 105 training rows against 200 predictors. OLS train MSE 0.0000 and test MSE
12.9587: perfect on the data it has seen, useless on data it has not.

<!-- idiom-note -->
> ### New idiom — pipelines
>
> A **pipeline** chains preprocessing and a model into one estimator:
>
> ```python
> model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
> model.fit(X_train, y_train)
> ```
>
> It still obeys the build → fit → ask pattern from Lab 4; there are simply
> two objects inside it now. Calling `.fit()` fits the scaler **and** the
> model; calling `.predict()` applies the fitted scaler first, then the model.
> `model[-1]` is the last object in the pipeline, the model itself.
>
> This is not a convenience. Scaling the whole dataset before splitting leaks
> information from the test set into training — the scaler would learn the
> test set's mean. A pipeline makes that mistake impossible, because the
> scaler is only ever fitted on whatever `.fit()` is given. Step 6 uses the
> same idea one level down: there the scaler is fitted again inside every
> cross-validation fold.

Step 3 fits the two penalised models, each in a pipeline with a scaler, and each choosing its own
penalty by 5-fold cross-validation. scikit-learn calls the penalty `alpha`; the chapter calls it
λ (lambda). They are the same number.

- `LassoCV(cv=5, max_iter=20_000)` tries 100 penalties, from the smallest one that zeroes every
  coefficient down to a thousandth of it, and keeps the one with the smallest CV error. `cv=5`
  deals the rows into 5 folds in order, without Lab 15's shuffle. `max_iter` lets the solver take
  more passes before it gives up. Inside LassoCV's own folds this scaler is fitted once, on all 105
  rows; Step 6 refits it per fold.
- `RidgeCV(alphas=np.logspace(-3, 3, 60), cv=5)` tries 60 penalties from 0.001 to 1,000.
  Without `cv=5`, RidgeCV uses a quick leave-one-out formula, and on a design with more
  predictors than rows that slides down to the smallest penalty in the grid, whatever it is: the
  "chosen" penalty is just the floor you typed. With `cv=5` it runs the same 5-fold search as
  LassoCV.
- After fitting, `.alpha_` is the penalty each one chose. A trailing underscore marks something
  learned from the data during `.fit()`.

In [ ]:
# GUIDED — run as-is
# Step 3: LASSO and Ridge, each with its penalty chosen by 5-fold CV
lasso = make_pipeline(StandardScaler(),
                      LassoCV(cv=5, max_iter=20_000)).fit(X_tr, y_tr)
ridge = make_pipeline(StandardScaler(),
                      RidgeCV(alphas=np.logspace(-3, 3, 60), cv=5)).fit(X_tr, y_tr)

print(f"chosen penalties:  LASSO alpha = {lasso[-1].alpha_:.4f}"
      f"   Ridge alpha = {ridge[-1].alpha_:.4f}")

**Expected output:** LASSO alpha = 0.1978, Ridge alpha = 1.4208. Neither is at the edge of its
grid, so both were actually chosen.

Step 4 scores all three models on the training rows and the test rows, and counts the
coefficients that are not zero. `m.coef_ if name == "OLS" else m[-1].coef_` picks OLS's
coefficients directly, and the model inside the pipeline for the other two.
`np.abs(coefs) > 1e-8` is True for each coefficient bigger than a hair, and `np.sum` of
True/False counts the Trues. In the last print, `:+.1%` shows a share as a percentage with its
sign, so −0.22 prints as −22.0%.

In [ ]:
# GUIDED — run as-is
# Step 4: training error, test error and the number of coefficients kept
# <12 left-aligns in 12 characters; >12.4f right-aligns a number in 12, with 4 decimals
print(f"{'model':<12}{'train MSE':>12}{'test MSE':>12}{'nonzero coefs':>16}")
print("-" * 52)
for name, m in [("OLS", ols), ("LASSO", lasso), ("Ridge", ridge)]:
    coefs = m.coef_ if name == "OLS" else m[-1].coef_
    print(f"{name:<12}{metrics.mean_squared_error(y_tr, m.predict(X_tr)):>12.4f}"
          f"{metrics.mean_squared_error(y_te, m.predict(X_te)):>12.4f}"
          f"{int(np.sum(np.abs(coefs) > 1e-8)):>16}")

ols_test = metrics.mean_squared_error(y_te, ols.predict(X_te))
print("\ntest MSE against unpenalised OLS:")
for name, m in [("Ridge", ridge), ("LASSO", lasso)]:
    t = metrics.mean_squared_error(y_te, m.predict(X_te))
    print(f"   {name:<6}{t:>10.4f}   {t / ols_test - 1:>+7.1%} vs OLS")

**Expected output:** OLS 0.0000 / 12.9587 / 200, LASSO 1.1002 / 1.4219 / 16, Ridge 0.0021 /
10.1128 / 200. Then Ridge −22.0% and LASSO −89.0% against OLS.

Note the ordering: OLS and Ridge both fit the training rows almost perfectly, and LASSO has the
worst training error of the three and by far the best test error. As in Lab 15, training error is
not where regularization shows up.

### Interpretation Questions

1. Both penalised models are *biased* by construction. Read Step 4's comparison with OLS and say what each penalty's bias bought, in the vocabulary of Lab 15's decomposition. Then explain why the same idea bought so much more for LASSO than for Ridge **on this particular design**.

*Your answer here:*

## Part 2: The Path (GUIDED — 16 min)

Step 5 fits LASSO at 60 penalties at once and draws every coefficient against the penalty: the
*path*. `StandardScaler().fit_transform(X_tr)` fits the scaler and scales in one call.
`lasso_path(Xs, y_tr, n_alphas=60)` runs from the smallest penalty that zeroes every coefficient
down to a thousandth of it, and gives back three things: the penalties, largest first; the
coefficients, one row per variable and one column per penalty; and one we do not need, which goes
to `_`, Python's name for "ignore this". `ax.invert_xaxis()` puts the largest penalty on the left.

In [ ]:
# GUIDED — run as-is
# Step 5: watch coefficients shrink as the penalty grows
Xs = StandardScaler().fit_transform(X_tr)
alphas, coefs, _ = lasso_path(Xs, y_tr, n_alphas=60)

fig, ax = plt.subplots(figsize=(8, 4.5))
for j in range(p):
    if j < k:                                   # the five true predictors
        ax.plot(np.log10(alphas), coefs[j], color="firebrick", lw=2)
    else:                                       # the 195 noise columns
        ax.plot(np.log10(alphas), coefs[j], color="silver", lw=0.6)
ax.set_xlabel("log10(penalty)")
ax.set_ylabel("coefficient")
ax.set_title("LASSO path — the five true predictors in red")
ax.invert_xaxis()
plt.tight_layout()
plt.savefig("figures/ch16_lasso_path.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** read the chart left to right. The penalty falls and variables enter, the
red ones first. Step 5, continued, finds where along the 60 penalties the five true predictors
are all in, and where the first noise variable enters. Position 0 is the largest penalty, where
every coefficient is zero. `.all(axis=0)` asks, for each penalty (each column), whether all of
the rows are True; `.any(axis=0)` whether any is. `np.argmax` on True/False gives the position of
the first True.

In [ ]:
# GUIDED — run as-is
# Step 5, continued: where the true five and the first noise variable enter the path
nonzero = coefs != 0                        # True where a coefficient is not zero
all_true_in = np.argmax(nonzero[:k].all(axis=0))
first_noise = np.argmax(nonzero[k:].any(axis=0))
print(f"all five true predictors are in from step {all_true_in} of 60 "
      f"(penalty {alphas[all_true_in]:.4f})")
print(f"the first noise variable enters at step {first_noise} "
      f"(penalty {alphas[first_noise]:.4f})")

**Expected output:** all five true predictors are in from step 6 (penalty 1.2954), and the first
noise variable enters at step 17 (penalty 0.3573).

Step 6 chooses the penalty by cross-validation once more, as Lab 15 chose a degree, but this time
a fresh scaler is fitted inside every fold, on that fold's training rows only. `GridSearchCV`
runs the whole loop in one call: for each penalty in the grid and each of the 5 folds, it fits a
new copy of the pipeline, scaler included, on the other four folds, and scores the fold left out.

- `np.logspace(-2, 0, 21)` makes 21 penalties from 10⁻² = 0.01 to 10⁰ = 1, evenly spaced in
  powers of ten.
- `folds` is Lab 15's deal: 5 folds, shuffled with a fixed seed.
- `{"lasso__alpha": LAMBDAS}` says which setting to vary. A pipeline names each step after its
  class in lower case, so `lasso__alpha` (two underscores) is "the `alpha` of the step called
  `lasso`".
- scikit-learn always keeps the *largest* score, so to minimise the MSE it scores by minus the
  MSE, `"neg_mean_squared_error"`. The minus signs below turn it back.
- `.cv_results_["mean_test_score"]` holds the score for each penalty, averaged over the 5 folds,
  in grid order. `.best_params_` and `.best_score_` are the winner and its score.

In [ ]:
# GUIDED — run as-is
# Step 6: choose lambda by 5-fold CV, with the scaler fitted inside every fold
LAMBDAS = np.logspace(-2, 0, 21)     # 21 penalties from 0.01 to 1
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(make_pipeline(StandardScaler(), Lasso()),
                      {"lasso__alpha": LAMBDAS},
                      cv=folds, scoring="neg_mean_squared_error").fit(X_tr, y_tr)

cv_mse = -search.cv_results_["mean_test_score"]     # one CV MSE per penalty
lam_cv = search.best_params_["lasso__alpha"]
cv_best = -search.best_score_
print(f"CV chooses lambda = {lam_cv:.4f}, with a CV MSE of {cv_best:.4f}")

**Expected output:** "CV chooses lambda = 0.1995, with a CV MSE of 1.5502". The AI-Assisted
Expansion at the end of this lab asks you to reproduce both numbers with a loop written by AI.

The lecture's workflow says to look at the CV curve, not just its minimum. Step 6, continued,
draws it.

In [ ]:
# GUIDED — run as-is
# Step 6, continued: the CV curve, with the chosen penalty dashed
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.log10(LAMBDAS), cv_mse, "o-", color="firebrick")
ax.axvline(np.log10(lam_cv), ls="--", color="gray")
ax.set_xlabel("log10(penalty)")
ax.set_ylabel("5-fold CV MSE")
plt.tight_layout()
plt.savefig("figures/ch16_cv_curve.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretation Questions

On this design the separation is clean: Step 5 found all five true predictors in by step 6, and
the first noise variable not until step 17. Step 2's design produces that gap: even the smallest
true coefficient, 1.5 in size, is large against noise with a standard deviation of 1, and the
columns are drawn independently, so they are nearly uncorrelated.

1. Describe what would have to change for a noise variable to enter *before* a true one, and what that says about trusting LASSO's selection on real data, where neither of those conditions holds.

*Your answer here:*

## Part 3: What It Recovered (GUIDED — 4 min)

Step 7 scores Step 3's LASSO against the truth, which only a simulation knows. `set(...)` makes a
set of variable numbers; `&` keeps the numbers in both sets, and `-` the ones in the first set
only. `np.flatnonzero(sel)` gives the positions where `sel` is True.

In [ ]:
# GUIDED — run as-is
# Step 7: selection accuracy against the known truth
sel = np.abs(lasso[-1].coef_) > 1e-8
true_set = set(range(k))
found = set(np.flatnonzero(sel))

print(f"chosen penalty          : {lasso[-1].alpha_:.4f}")
print(f"variables selected      : {len(found)}")
print(f"true positives          : {len(found & true_set)} of {k}")
print(f"false positives         : {len(found - true_set)}")
print()
print(f"{'variable':>10}{'true beta':>12}{'LASSO':>10}")
print("-" * 34)
for j in range(k):
    print(f"{j:>10}{beta[j]:>12.3f}{lasso[-1].coef_[j]:>10.3f}")

**Expected output:** penalty 0.1978, 16 variables selected, 5 of 5 true positives and 11 false
positives. Then the five true coefficients beside LASSO's: 3.000 and 2.524, −2.000 and −1.769,
1.500 and 1.018, 2.500 and 1.971, −1.800 and −1.384.

## Part 4: Correlated Predictors (YOUR TASK — 12 min)

Step 8 builds a design where two predictors are nearly identical and *both* matter. It copies
`X`, replaces column 5 with column 0 plus a little noise, and gives column 5 a true coefficient of
2.0. `np.corrcoef(a, b)[0, 1]` is the correlation between `a` and `b`. It then splits the new
design the way Step 2 split the first one.

In [ ]:
# GUIDED — run as-is
# Step 8: two near-duplicate predictors, both real
Xc = X.copy()
Xc[:, 5] = Xc[:, 0] + rng.normal(0, 0.05, n)     # a near-copy of a true one
beta_c = beta.copy()
beta_c[5] = 2.0
yc = Xc @ beta_c + rng.normal(0, 1.0, n)

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3,
                                              random_state=RANDOM_STATE)
print(f"correlation between variables 0 and 5: {np.corrcoef(Xc[:, 0], Xc[:, 5])[0, 1]:.4f}")
print(f"true coefficients: var 0 = {beta_c[0]:.1f}, var 5 = {beta_c[5]:.1f}")

**Expected output:** a correlation of 0.9985, and true coefficients of 3.0 and 2.0.

**Your task.** Fill in the two blanks in Step 9: the LASSO and the Ridge model of Step 3, fitted
to this design's training rows. The print below them then shows each model's coefficients on
variables 0 and 5, and their sum.

In [ ]:
# YOUR TASK — fill in the two blanks (____), then run
# Step 9: LASSO and Ridge on the near-duplicates
lasso_c = ____      # Step 3's LASSO pipeline, fitted to Xc_tr and yc_tr
ridge_c = ____      # Step 3's Ridge pipeline, fitted the same way

print(f"{'model':<8}{'var 0':>9}{'var 5':>9}{'sum':>9}")
for name, m in [("LASSO", lasso_c), ("Ridge", ridge_c)]:
    c = m[-1].coef_
    print(f"{name:<8}{c[0]:>9.3f}{c[5]:>9.3f}{c[0] + c[5]:>9.3f}")

**Expected output:** LASSO 4.462, 0.000, sum 4.462; Ridge 1.618, 1.609, sum 3.226. If Ridge
prints 1.598 and 1.590, your RidgeCV has no `cv=5`. If LASSO prints 4.353, you fitted on all 150
rows, `Xc` and `yc`, rather than the training rows.

### Interpretation Questions

1. LASSO kept one of the pair and dropped the other; Ridge split the weight between them. Which behaviour do you want when the two columns measure the same underlying thing, and which when they are two different things? Say why.

*Your answer here:*

---

## Extension (Optional — 20 min): Elastic Net

There is no starter code: write it in the cell below.

`ElasticNetCV` (from `sklearn.linear_model`) mixes the two penalties. Its `l1_ratio` sets the
mix: 1 is pure LASSO, and values near 0 are nearly pure Ridge. Fit it in Step 3's pipeline on
Step 8's training rows, `Xc_tr` and `yc_tr`, and let it choose among several mixes, for example
`l1_ratio=[.1, .5, .9, 1]`, with `cv=5`. Compare its test MSE and its coefficients on variables 0
and 5 with `lasso_c` and `ridge_c` from Step 9.

**Your answer.** Which wins on test error, and which recovers the near-duplicate pair most
sensibly? They may not be the same model.

In [ ]:
# EXTENSION (optional) — write your Elastic Net here

*Your answer here:*

---

## Challenge (Take-Home): Standardisation Is Not Optional

There is no starter code: write your answer in the cell below and in new cells under it.

1. Multiply one true predictor of `X` by 1,000, so its scale dwarfs the rest, and split it as
   Step 2 did.
2. Fit LASSO *without* `StandardScaler` and look at what it keeps.
3. Refit with the scaler and compare.
4. Explain why the penalty term makes scaling a question of correctness rather than preference.
   Then fit OLS with and without the rescaled column. With fewer rows than predictors, OLS picks
   the smallest-coefficient fit among many that fit perfectly (Lab 15's Challenge), so it is not
   quite indifferent to the scale either: say why.

In [ ]:
# YOUR TASK — take-home Challenge: write your code here, and in new cells below

---

## AI-Assisted Expansion: a cross-validation loop over λ, your prompt, checked by you

Step 6 chose λ in one call: `GridSearchCV` ran the loop out of sight, over every penalty and every
fold, with a scaler fitted inside each fold. Here you ask AI to write that loop out in full, as a
function. Loop code like this is quick for AI to write and easy to get subtly wrong: every choice
your prompt leaves out, the AI makes for you, and most of those choices change the answer. This
time no prompt is given. You write it.

**The target.** Your function must reproduce Step 6: **λ = 0.1995, with a CV MSE of 1.5502.**
Both numbers, not just λ. On this design, loops that differ in important ways often land on the
same λ; the CV MSE is what tells them apart.

**What the check calls.** `cv_lasso(X, y, lambdas)`, where `X` is a NumPy array of predictors,
`y` the outcome and `lambdas` an array of penalties. It must give back two numbers: the chosen λ,
then its CV MSE. The check calls it twice: on `X_tr`, `y_tr` and Step 6's `LAMBDAS`, and on the
same data with every second penalty, `LAMBDAS[::2]`, where it compares against Step 6's curve at
those penalties.

**What a good prompt states.** The prompt is yours. Before you send it, make sure it answers each
of these. Step 6 holds every choice; Step 3 made different ones.
- the inputs and the output: the function's name, its arguments in order, their types, and what
  it gives back;
- an explicit loop over the penalties and the folds, not `GridSearchCV` or `LassoCV`;
- the penalties: which values to try;
- the folds: how many, whether they are shuffled, and with what seed;
- the scaling: what gets standardised, and which rows the scaler is fitted on;
- the model: which scikit-learn estimator, and what it calls the penalty;
- the score: which error each fold reports, and how the folds' errors are combined;
- how the best penalty is chosen;
- what to leave out (example data, plots), and a comment on each line.

**What to do**
1. Write your round 1 prompt in the cell below, then send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read what comes back before you run anything. Paste only the function into the round 1 cell,
   and run the check below it.
3. If a pair says `NO`, find the choice in the function that caused it. Write a round 2 prompt
   that states that choice, ask again, paste the new function into the round 2 cell, and run its
   check. A third round is allowed: paste it over round 2's function and run the same check.
   Stop when every pair says `yes` and you can explain each line, or when you decide to stop.
4. If every pair says `yes` in round 1, use round 2 to state one choice the function made that
   your prompt never asked for, and check that it still matches.
5. Fill in the checking log and answer the two questions. Your prompts, the log and the answers
   are what is graded, not the code.

**Reading the check.** Each line shows your Step 6 number, the AI's, `same?`, and how far apart
they are in percent. Both come from the same data in the same session, so a correct function
agrees in every printed digit.

**The one allowed gap is the solver's.** `Lasso` stops once it is close enough to the exact
answer, so a loop that starts each fit from the last one (`warm_start=True`, or `lasso_path`) can
land a few hundredths of a percent at most away, with λ identical. To test it, ask for a version
that builds a new `Lasso` for every penalty; if it matches in every digit, the gap was the solver.

Keep the function's name and the order of its arguments. Paste only the function: if the AI also
wrote example data, leave it out, because it can replace the lab's data.

In [ ]:
# GUIDED — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves copies of the data and the penalties now, before anything is pasted.
saved_X, saved_y, saved_lambdas = X_tr.copy(), y_tr.copy(), LAMBDAS.copy()

# Step 6's numbers. [::2] takes every second item, from position 0: the same penalties in
# LAMBDAS[::2] and the same points of Step 6's curve in cv_mse[::2]
best_of_half = int(np.argmin(cv_mse[::2]))
targets = [("all 21 penalties", saved_lambdas, lam_cv, cv_best),
           ("every second one", saved_lambdas[::2],
            saved_lambdas[::2][best_of_half], cv_mse[::2][best_of_half])]


def lab_data_unchanged():
    """True if X_tr, y_tr and LAMBDAS still hold what the lab made."""
    # try runs the line under it; if that raises an error (say X_tr is gone), except runs instead
    try:
        return (np.array_equal(X_tr, saved_X) and np.array_equal(y_tr, saved_y)
                and np.array_equal(LAMBDAS, saved_lambdas))
    except Exception:
        return False


def show(label, mine, ai):
    """One line: your number, the AI's, whether they agree, and how far apart they are."""
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    same = "yes" if np.isclose(mine, ai, rtol=1e-9, atol=0) else "NO"
    pct = 100 * (ai - mine) / abs(mine)
    # a gap under 1%: sign and 2 significant digits, so a tiny one prints like +9.6e-06;
    # otherwise a whole number with its sign, like -200
    apart = f"{pct:+.2g}" if abs(pct) < 1 else f"{pct:+.0f}"
    print(f"{label:<28}{mine:>10.4f}{ai:>10.4f}{same:>7}{apart:>11}%")


def plain(value):
    """A number as 4 decimals, or the name of its type if it is not a number."""
    try:
        return f"{float(value):.4f}"
    except Exception:
        return type(value).__name__


def check_cv_lasso(fn):
    """Run the AI's cv_lasso on X_tr and y_tr, with two sets of penalties, beside Step 6."""
    if not lab_data_unchanged():
        print("The pasted code changed X_tr, y_tr or LAMBDAS. "
              "Paste only the function, then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("cv_lasso should be a function. Paste the function the AI wrote.")
        return
    for label, lambdas, lam_mine, cv_mine in targets:
        # copies, so a function that changes its inputs cannot change the lab's data
        try:
            result = fn(saved_X.copy(), saved_y.copy(), lambdas.copy())
        except Exception as err:
            # type(err).__name__ is the kind of error, such as TypeError
            print(f"With {label}, the AI's function stopped with an error. "
                  f"{type(err).__name__}: {err}")
            return
        try:
            lam_ai, cv_ai = result               # fails unless there are exactly two things
            lam_ai, cv_ai = float(lam_ai), float(cv_ai)
        except Exception:
            if isinstance(result, dict):
                shown = ", ".join(f"{key}: {plain(v)}" for key, v in result.items())
            elif isinstance(result, (tuple, list)):
                shown = ", ".join(plain(v) for v in result)
            else:
                shown = plain(result)
            print(f"cv_lasso gave back a {type(result).__name__}: {shown}")
            print("The check needs two numbers: the chosen lambda, then its CV MSE.")
            return
        if label == targets[0][0]:               # the column headings, once
            print(f"{'':<28}{'yours':>10}{'AI':>10}{'same?':>7}{'% apart':>12}")
        show(f"{label}: lambda", lam_mine, lam_ai)
        show(f"{label}: CV MSE", cv_mine, cv_ai)
    print("\nIf a pair differs, find out why before you trust the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the cv_lasso function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data, leave it out.

In [ ]:
# CHECK — round 1
# globals() holds every name defined so far in this notebook
if "cv_lasso" not in globals():
    print("Paste the AI's cv_lasso function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    cv_lasso_r1 = cv_lasso      # kept, so the round 2 check can tell the two apart
    check_cv_lasso(cv_lasso_r1)

**Your prompt, round 2** (and round 3 under it, if you need one). Keep what worked in round 1,
and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the cv_lasso function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data, leave it out.

In [ ]:
# CHECK — round 2 (and round 3)
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "cv_lasso" not in globals() or cv_lasso is globals().get("cv_lasso_r1"):
    print("Paste the AI's round 2 cv_lasso function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    check_cv_lasso(cv_lasso)

### Checking log

This is graded, with your prompts and the two answers below. One row per round (a third round,
if any, goes in round 2's row): what you checked, how you checked it, and what you found. A row
that names the pair that differed, how far apart it was, and the line of the AI's code
responsible earns credit; "it worked" does not.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |

### Questions

1. **Critique, in your own words.** Name one choice the AI made that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. (A choice that caused a `NO` counts.) Then quote the lines of your final function that fit the scaler and the model inside one fold, or the line that hands `cross_val_score` the pipeline, and explain in 2-3 sentences, in your own words, what they do and why that refits the scaler per fold.
2. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not, trust that function to choose λ on data the check has never seen? Then: when would you ask AI to write a cross-validation loop like this, and when would you write it yourself or use `GridSearchCV`?

*Your answers here:*

1.
2.

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Built a simulated design with 200 predictors, only 5 of them real,
  and 105 training rows, and fitted OLS, LASSO and Ridge to it
* Compared test MSEs: OLS [YOUR VALUE], Ridge [YOUR VALUE], LASSO
  [YOUR VALUE]
* Traced the LASSO path: all five true predictors were in from step
  [YOUR VALUE], and the first noise variable entered at step
  [YOUR VALUE]
* Chose the penalty by 5-fold cross-validation with the scaler fitted
  inside every fold: lambda [YOUR VALUE], CV MSE [YOUR VALUE]
* Scored LASSO's selection against the known truth: [YOUR VALUE] of 5
  true predictors found, [YOUR VALUE] false positives
* Gave LASSO and Ridge two near-duplicate predictors: LASSO put
  [YOUR VALUE] on one and zero on the other; Ridge split the weight
  [YOUR VALUE] and [YOUR VALUE]
* Wrote my own prompt asking an AI for the cross-validation loop, went
  [YOUR VALUE] rounds, and checked its lambda and CV MSE against mine:
  [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Regularization: Buying Bias to Sell Variance
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab16-regularization`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab16-regularization`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab16-regularization`, branch `main`,
   commit message `Lab 16 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.